# Solution 04 — the temperature agent

Two exact-conversion tools, one agent that *must* use them, and proof from the
message history that the tools really ran.

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.messages import ToolReturnPart

temp_agent = Agent(
    MODEL,
    instructions=(
        "You compare temperatures. You NEVER do arithmetic yourself — always convert "
        "with your tools first, then reason about the results."
    ),
)


@temp_agent.tool_plain
def c_to_f(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit, exactly."""
    return celsius * 9 / 5 + 32


@temp_agent.tool_plain
def f_to_c(fahrenheit: float) -> float:
    """Convert a temperature from Fahrenheit to Celsius, exactly."""
    return (fahrenheit - 32) * 5 / 9


result = await temp_agent.run("Is 25°C warmer than 80°F? Show both on one scale.")
print(result.output)

tools_used = [
    part.tool_name
    for message in result.all_messages()
    for part in message.parts
    if isinstance(part, ToolReturnPart)
]
print("tools that actually ran:", tools_used)

**Notes**

- 25°C = 77°F, so 25°C is *cooler* than 80°F — a classic model-arithmetic trap,
  which is exactly why the conversion lives in your tools.
- If your agent tried to answer without tools, that's the `instructions` string not
  being firm enough — this is normal prompt hardening, not a framework failure.